# 02 — RL Fine-tuning

Loads the IL checkpoint, sets up the K-value map, and runs `train_rl()`.  
**Run `01_setup.ipynb` first** so that `DATA_DIR`, `CLUSTER_DIR`, `IL_WEIGHTS`, `RL_SAVE`, etc. are defined.

In [ ]:
# Re-run setup (or %run 01_setup.ipynb if variables aren't in scope)
import sys, os
sys.path.insert(0, '..')

import copy, random
import numpy as np
import torch

from src.config import (
    DEVICE,
    RL_EPOCHS, RL_LR, RL_ENTROPY_COEF, RL_KL_COEF,
    RL_EVAL_EVERY, RL_PATIENCE, RL_TEMPERATURE,
    RL_LAMBDA_WEIGHT_PENALTY, RL_LAMBDA_VOLUME_PENALTY,
)
from src.model import TransformerClusterer
from src.packer import pd3Packer
from src.train_rl import train_rl

# These must be set — either run 01_setup.ipynb or define them here
# DATA_DIR    = '../good_data'
# CLUSTER_DIR = '../clustering_v2'
# IL_WEIGHTS  = os.path.join(CLUSTER_DIR, 'transformer_imitation_v2.pt')
# RL_SAVE     = os.path.join(CLUSTER_DIR, 'transformer_rl_v2_K.pt')
# RL_BASELINE_CACHE = os.path.join(CLUSTER_DIR, 'il_baseline_cache_K.pkl')
# RL_LOG      = os.path.join(CLUSTER_DIR, 'rl_training_log_v2_K.csv')

In [ ]:
# Load IL weights into RL model
model = TransformerClusterer().to(DEVICE)

assert os.path.exists(IL_WEIGHTS), (
    f'IL checkpoint not found at {IL_WEIGHTS}. '
    f'Train the IL model first or check CLUSTER_DIR.'
)

ckpt  = torch.load(IL_WEIGHTS, map_location='cpu', weights_only=False)
state = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
model.load_state_dict(state, strict=True)

il_epoch    = ckpt.get('epoch',        '?') if isinstance(ckpt, dict) else '?'
il_val_loss = ckpt.get('val_loss',     float('nan')) if isinstance(ckpt, dict) else float('nan')
print(f'IL checkpoint loaded: epoch={il_epoch}, val_loss={il_val_loss:.4f}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

il_model = copy.deepcopy(model).to(DEVICE)
il_model.eval()
for p in il_model.parameters():
    p.requires_grad_(False)
print('Frozen IL reference model ready.')

In [ ]:
import pandas as pd

# Build K-value map: assign each instance one of the spread-penalty coefficients
K_VALUES = [100, 500, 1000, 3000, 5000]

train_meta_df = pd.read_csv(os.path.join(DATA_DIR, 'synthetic_train', 'metadata.csv'))
test_meta_df  = pd.read_csv(os.path.join(DATA_DIR, 'synthetic_test',  'metadata.csv'))

k_values_map_dict = {}

for meta_df, split_name in [(train_meta_df, 'train'), (test_meta_df, 'test')]:
    instances = meta_df['instance'].tolist()
    random.shuffle(instances)
    per_k = len(instances) // len(K_VALUES)
    for i, k_val in enumerate(K_VALUES):
        for tag in instances[i * per_k:(i + 1) * per_k]:
            k_values_map_dict[tag] = k_val

print(f'Assigned K values to {len(k_values_map_dict)} instances.')

In [ ]:
history = train_rl(
    model                  = model,
    il_model               = il_model,
    data_dir               = DATA_DIR,
    n_epochs               = RL_EPOCHS,
    lr                     = RL_LR,
    entropy_coef           = RL_ENTROPY_COEF,
    kl_coef                = RL_KL_COEF,
    eval_every             = RL_EVAL_EVERY,
    patience               = RL_PATIENCE,
    save_path              = RL_SAVE,
    log_path               = RL_LOG,
    il_baseline_cache_path = RL_BASELINE_CACHE,
    device                 = DEVICE,
    temperature            = RL_TEMPERATURE,
    max_instances          = None,   # full dataset
    packer                 = pd3Packer(),   # swap for EPIPacker() if py3dbp not installed
    lambda_weight_penalty  = RL_LAMBDA_WEIGHT_PENALTY,
    lambda_volume_penalty  = RL_LAMBDA_VOLUME_PENALTY,
    k_values_map_dict      = k_values_map_dict,
)